In [1]:
!pip install protobuf==3.20.3 --quiet
!pip install transformers bitsandbytes accelerate --quiet

import importlib, site, sys
importlib.reload(site)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not 

<module 'site' (frozen)>

In [2]:
import torch
import time
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import csv
import sys
import re
from tqdm.notebook import tqdm
import os
from kaggle_secrets import UserSecretsClient

2025-11-14 00:23:34.691935: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763079814.883128      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763079814.935430      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Hugging Face token loaded from Kaggle Secrets.")
except Exception as e:
    print("Could not load token from Kaggle Secrets. Trying environment variable.")
    hf_token = os.environ.get("HF_TOKEN") 

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading tokenizer for: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Hugging Face token loaded from Kaggle Secrets.
Loading tokenizer for: mistralai/Mistral-7B-Instruct-v0.3


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [4]:
print(f"Loading model: {model_id}")
model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config, 
        device_map="auto",
        token=hf_token
    )
print("Model loaded")

print("Creating text-generation pipeline...")

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)
print("Pipeline created successfully.")

Loading model: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded
Creating text-generation pipeline...
Pipeline created successfully.


In [5]:
file_path = '/kaggle/input/data-clean2/test_split.csv' 
example_path = '/kaggle/input/data-clean2/val_split.csv'

test_df = pd.read_csv(file_path)
print(test_df.head())

example_df = pd.read_csv(example_path)
print(example_df.head())

                                            Dialogue  Manipulative  \
0  Judge\r\nI understand, but my hypothetical ass...             0   
1  Judge:\nWell, but if -- why -- your voluntary ...             0   
2  Judge: Why did you go?\nPlaintiff: I had nothi...             1   
3  \nJudge:\nTell me what happened.\n\nPlaintiff ...             1   
4  Judge: Is your position— is there any daylight...             0   

  Primary Manipulator                            Manipulation Techniques  
0                 NaN                                                NaN  
1                 NaN                                                NaN  
2           Plaintiff                           deflection, minimization  
3           Defendant  gaslighting, deflection, emotional appeal, pla...  
4                 NaN                                                NaN  
                                            Dialogue  Manipulative  \
0  Judge: Justice Alito mentioned the Reich case ...       

In [6]:
allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion",
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

def format_example(example):
    return f"""Dialogue: {example['dialogue']}
Manipulation Present: {example['manipulative']}
Primary Manipulator: {example['manipulator']}
Techniques: {example['techniques']}"""


def generate_prompt(statement, few_shot_examples=None):
    prompt = f"""You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Determine if any manipulation is present or not.
3. If manipulation is present then identify the primary manipulator.
4. If manipulation is present identify the techniques used from the allowed list.

IMPORTANT: You can ONLY use these exact techniques (no variations or additions):
{', '.join(allowed_tactics)}
"""

    if few_shot_examples:
        prompt += "\nHere are some examples of how to analyze dialogues:\n\n"
        for example in few_shot_examples:
            prompt += format_example(example) + "\n\n"

    prompt += f"""\nNow analyze this new dialogue:\n\nDialogue: {statement}

Question: Is there manipulation present in this dialogue? If yes then identify the primary manipulator and techniques.

Response Format:
Manipulation Present: [Yes/No]
Primary Manipulator: [Plaintiff/Defendant/None]
Techniques: [technique1, technique2, ... from allowed_tactics only/None]"""

    messages = [{"role": "user", "content": prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def detect_manipulation(statement, llm_pipeline,few_shot_examples=None):
    prompt = generate_prompt(statement, few_shot_examples)
    terminators = [
        llm_pipeline.tokenizer.eos_token_id,
    ]
    response = llm_pipeline(
        prompt,
        max_new_tokens=150, 
        temperature=0.3,   
        do_sample=True,
        eos_token_id=terminators,
        pad_token_id=llm_pipeline.tokenizer.eos_token_id
    )

    full_text = response[0]["generated_text"]

    prompt_end_index = full_text.find(prompt.split("[/INST]")[-1].strip()) + len(prompt.split("[/INST]")[-1].strip())
  
    model_output = full_text[prompt_end_index:].strip() if prompt_end_index > len(prompt.split("[/INST]")[-1].strip()) else full_text.split("[/INST]")[-1].strip()
    return model_output

In [7]:
def extract_analysis_result(response):
    Manipulative_Pred = 0
    primary_manipulator = "None"
    techniques = []

    present_match = re.search(r"Manipulation Present:\s*(Yes|No)", response, re.IGNORECASE)
    if present_match and present_match.group(1).lower() == 'yes':
        Manipulative_Pred = 1
    else:
         print(f"Final extracted values: Manipulative={Manipulative_Pred}, Manipulator={primary_manipulator}, Techniques=None")
         print("-" * 80)
         return Manipulative_Pred, primary_manipulator, "None"

    # 2. Trích xuất Primary Manipulator (chỉ khi Manipulative_Pred=1)
    manipulator_match = re.search(r"Primary Manipulator:\s*([^\n]+)", response, re.IGNORECASE)
    if manipulator_match:
        manipulator_text = manipulator_match.group(1).strip().lower()
        if "plaintiff" in manipulator_text:
            primary_manipulator = "Plaintiff"
        elif "defendant" in manipulator_text:
            primary_manipulator = "Defendant"
  
    # 3. Trích xuất Techniques (chỉ khi Manipulative_Pred=1)
    techniques_match = re.search(r"Techniques:\s*([^\n]+)", response, re.IGNORECASE)
    if techniques_match:
        techniques_text = techniques_match.group(1).strip()
       
        if techniques_text.lower() not in ["none", "n/a", "", "[]"]:
          
             raw_techniques = [t.strip().lower() for t in techniques_text.split(',')]
           
             allowed_lower = [a.lower() for a in allowed_tactics]
             for raw_tech in raw_techniques:
                 if raw_tech in allowed_lower:
                   
                     original_tactic_index = allowed_lower.index(raw_tech)
                     techniques.append(allowed_tactics[original_tactic_index])
           

    techniques_list_or_none = techniques if techniques else ["None"] 

    print(f"Final extracted values: Manipulative={Manipulative_Pred}, Manipulator={primary_manipulator}, Techniques={techniques_list_or_none}")
    print("-" * 80)

    return Manipulative_Pred, primary_manipulator, techniques_list_or_none 

In [8]:
def process_dialogues(df, llm_pipeline, few_shot_examples=None):
    manipulative_preds, manipulators, all_techniques = [], [], []

    print("\nStarting analysis...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing dialogues"):

        try:
            response_text = detect_manipulation(row["Dialogue"], llm_pipeline, few_shot_examples)
            manipulative, primary_manipulator, techniques_list = extract_analysis_result(response_text)

            manipulative_preds.append(manipulative)
            manipulators.append(primary_manipulator)
            all_techniques.append(techniques_list)

        except Exception as e:
            print(f"\n!!! LỖI xử lý hàng {idx}: {e}")
            print("    Thêm giá trị mặc định (0, None, ['None'])")
            manipulative_preds.append(0) # Hoặc giá trị lỗi khác
            manipulators.append("Error")
            all_techniques.append(["Error"])

        # (Tùy chọn) In tiến trình - Vẫn nằm trong vòng lặp
        if (idx + 1) % 5 == 0:
            print(f"\nProcessed {idx + 1}/{len(df)} dialogues")

    print("\nProcessing complete! Assigning results to DataFrame...")

    if len(manipulative_preds) == len(df.index) and \
       len(manipulators) == len(df.index) and \
       len(all_techniques) == len(df.index):

        df["Manipulative_Pred"] = manipulative_preds
        df["Primary_Manipulator_Pred"] = manipulators
        techniques_series = pd.Series(all_techniques)
        df["Techniques_Pred"] = techniques_series.apply(lambda x: ', '.join(x) if isinstance(x, list) else "None")
        print("Successfully assigned results to DataFrame columns.")

    return df

def save_results(df, output_path):
    """Save the results to a CSV file and print summary"""
    try:
        df.to_csv(output_path, index=False)
        print(f"\nResults saved to {output_path}")

        print("\nAnalysis Summary:")
        print(f"Total dialogues analyzed: {len(df)}")
        if "Manipulative_Pred" in df.columns:
             print(f"Dialogues predicted with manipulation: {df['Manipulative_Pred'].sum()}")
             print(f"Dialogues predicted without manipulation: {len(df) - df['Manipulative_Pred'].sum()}")
        if "Primary_Manipulator_Pred" in df.columns:
            print("\nPredicted Manipulator Distribution:")
            print(df['Primary_Manipulator_Pred'].value_counts())
        if "Techniques_Pred" in df.columns:
             print("\nMost Common Predicted Techniques:")
             technique_series = df['Techniques_Pred'].str.split(', ').explode()
             technique_counts = technique_series[~technique_series.isin(['None', 'Error'])].value_counts()
             print(technique_counts.head(10))

    except Exception as e:
        print(f"!!! LỖI khi lưu file hoặc tạo tóm tắt: {e}")

In [9]:
def main():
    if 'test_df' in globals() and isinstance(test_df, pd.DataFrame) and not test_df.empty:
        print("Starting main processing function...")
        few_shot_examples = []
        for _, row in example_df.iterrows():
            example = {
                "dialogue": row['Dialogue'],
                "manipulative": "Yes" if row['Manipulative'] == 1 else "No",
                "manipulator": row['Primary Manipulator'],
                "techniques": row['Manipulation Techniques'] if pd.notna(row['Manipulation Techniques']) else "None"
            }
            few_shot_examples.append(example)
        few_shot_examples = few_shot_examples[:5]
        results_df = process_dialogues(test_df.copy(), llm_pipeline, few_shot_examples)
        output_path = "/kaggle/working/manipulation_results.csv"
        save_results(results_df, output_path)
        print("Main function finished.")
    else:
        print("!!! LỖI: test_df không được định nghĩa hoặc bị rỗng. Vui lòng chạy lại cell tải dữ liệu.")

if 'test_df' in globals(): 
    main()
else:
    print("Skipping main() call as test_df is not loaded.")

Starting main processing function...

Starting analysis...


Processing dialogues:   0%|          | 0/155 [00:00<?, ?it/s]

Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'playing the victim', 'deflection', 'gaslighting']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------

Processed 5/155 dialogues
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
----------------------------------------------------

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'framing the narrative', 'dismissal']
--------------------------------------------------------------------------------

Processed 10/155 dialogues
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
----------------------------------------------------------------

In [10]:
try:
    test = pd.read_csv('/kaggle/working/manipulation_results.csv')
    print("File loaded successfully.")

    # --- Phần 1: Phân loại nhị phân (Có/Không Thao túng) ---
    print("\n--- Evaluating Binary Classification ---")
    targets_bin = test["Manipulative"].astype(int).tolist()
   
    test["Manipulative_Pred"] = test["Manipulative_Pred"].astype(str)
    test["Manipulative_Pred_Mapped"] = test["Manipulative_Pred"].str.lower().map({'1': 1, '0': 0, 'yes': 1, 'no': 0})
    test["Manipulative_Pred_Filled"] = test["Manipulative_Pred_Mapped"].fillna(0)
    manipulative_preds = test["Manipulative_Pred_Filled"].astype(int).tolist()

    accuracy_bin = accuracy_score(targets_bin, manipulative_preds)
    precision_bin = precision_score(targets_bin, manipulative_preds, zero_division=0)
    recall_bin = recall_score(targets_bin, manipulative_preds, zero_division=0)
    f1_bin = f1_score(targets_bin, manipulative_preds, average="binary", zero_division=0)
    conf_matrix_bin = confusion_matrix(targets_bin, manipulative_preds)

    print("\nBinary Performance Metrics:")
    print(f"- Accuracy: {accuracy_bin:.3f}")
    print(f"- Precision: {precision_bin:.3f}")
    print(f"- Recall: {recall_bin:.3f}")
    print(f"- F1 Score: {f1_bin:.3f}")
    print(f"- Confusion Matrix:\n{conf_matrix_bin}")

    # --- Phần 2: Phân loại đa lớp (Người Thao túng Chính) ---
    print("\n--- Evaluating Multi-class Classification ---")
  
    test['Primary Manipulator'] = test['Primary Manipulator'].fillna('None').str.lower().str.capitalize()
    test['Primary_Manipulator_Pred'] = test['Primary_Manipulator_Pred'].fillna('None').str.lower().str.capitalize()

    all_categories = sorted(list(set(test["Primary Manipulator"].tolist() + test["Primary_Manipulator_Pred"].tolist())))
    encoder = LabelEncoder()
    encoder.fit(all_categories)

    targets_encoded = encoder.transform(test["Primary Manipulator"])
    preds_encoded = encoder.transform(test["Primary_Manipulator_Pred"])

    label_mapping_multi = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print("\nLabel Encoding Mapping:", label_mapping_multi)

    accuracy_multi = accuracy_score(targets_encoded, preds_encoded)
    precision_multi_w = precision_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
    recall_multi_w = recall_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
    f1_multi_w = f1_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
    precision_multi_m = precision_score(targets_encoded, preds_encoded, average="macro", zero_division=0)
    recall_multi_m = recall_score(targets_encoded, preds_encoded, average="macro", zero_division=0)
    f1_multi_m = f1_score(targets_encoded, preds_encoded, average="macro", zero_division=0)
    conf_matrix_multi = confusion_matrix(targets_encoded, preds_encoded, labels=encoder.transform(encoder.classes_))

    print("\nMulti-class Performance Metrics:")
    print(f"- Accuracy: {accuracy_multi:.3f}")
    print(f"- Precision (Weighted): {precision_multi_w:.3f}")
    print(f"- Recall (Weighted): {recall_multi_w:.3f}")
    print(f"- F1 Score (Weighted): {f1_multi_w:.3f}")
    print(f"- Precision (Macro): {precision_multi_m:.3f}")
    print(f"- Recall (Macro): {recall_multi_m:.3f}")
    print(f"- F1 Score (Macro): {f1_multi_m:.3f}")
    print(f"- Confusion Matrix (Labels: {encoder.classes_}):\n{conf_matrix_multi}")

    # --- Phần 3: Phân loại đa nhãn (Kỹ thuật) ---
    print("\n--- Evaluating Multi-label Classification ---")
    test["Manipulation Techniques"] = test["Manipulation Techniques"].fillna("none").astype(str)
    test["Techniques_Pred"] = test["Techniques_Pred"].fillna("none").astype(str)

    def preprocess_techniques(tech_string):
        if pd.isna(tech_string) or tech_string.strip().lower() in ['none', '', 'nan']:
            return ["none"]
        techniques = [t.strip().lower() for t in tech_string.split(',') if t.strip()]
        return techniques if techniques else ["none"]

    test["Manipulation Techniques_List"] = test["Manipulation Techniques"].apply(preprocess_techniques)
    test["Techniques_Pred_List"] = test["Techniques_Pred"].apply(preprocess_techniques)

    unique_gt_labels = set(label for labels in test["Manipulation Techniques_List"] for label in labels)
    unique_pred_labels = set(label for labels in test["Techniques_Pred_List"] for label in labels)
    all_possible_labels = sorted(list(unique_gt_labels.union(unique_pred_labels)))
    if "none" in all_possible_labels:
        all_possible_labels.remove("none")

    if all_possible_labels: 
        mlb = MultiLabelBinarizer()
        mlb.fit([all_possible_labels])
        y_true = mlb.transform(test["Manipulation Techniques_List"])
        y_pred = mlb.transform(test["Techniques_Pred_List"])

        print("\nClasses for multi-label:", mlb.classes_)

        accuracy_ml = accuracy_score(y_true, y_pred) # Exact Match Ratio
        precision_ml_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
        recall_ml_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
        f1_ml_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
        precision_ml_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
        recall_ml_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
        f1_ml_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        jaccard_ml = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

        print("\nMulti-label Performance Metrics:")
        print(f"- Exact Match Ratio (Accuracy): {accuracy_ml:.4f}")
        print(f"- Precision (Macro Avg): {precision_ml_macro:.4f}")
        print(f"- Recall (Macro Avg): {recall_ml_macro:.4f}")
        print(f"- F1 Score (Macro Avg): {f1_ml_macro:.4f}")
        print(f"- Precision (Micro Avg): {precision_ml_micro:.4f}")
        print(f"- Recall (Micro Avg): {recall_ml_micro:.4f}")
        print(f"- F1 Score (Micro Avg): {f1_ml_micro:.4f}")
        print(f"- Jaccard Score (Samples Avg): {jaccard_ml:.4f}")
    else:
        print("\nNo valid manipulation techniques found (only 'none'), skipping multi-label metrics calculation.")

except FileNotFoundError:
    print(f"Error: File not found at '/kaggle/working/manipulation_results.csv'.")
except Exception as e:
    print(f"An unexpected error occurred during evaluation: {e}")
    import traceback
    traceback.print_exc()

File loaded successfully.

--- Evaluating Binary Classification ---

Binary Performance Metrics:
- Accuracy: 0.787
- Precision: 0.970
- Recall: 0.674
- F1 Score: 0.795
- Confusion Matrix:
[[58  2]
 [31 64]]

--- Evaluating Multi-class Classification ---

Label Encoding Mapping: {'Defendant': 0, 'None': 1, 'Plaintiff': 2}

Multi-class Performance Metrics:
- Accuracy: 0.574
- Precision (Weighted): 0.726
- Recall (Weighted): 0.574
- F1 Score (Weighted): 0.526
- Precision (Macro): 0.663
- Recall (Macro): 0.579
- F1 Score (Macro): 0.515
- Confusion Matrix (Labels: ['Defendant' 'None' 'Plaintiff']):
[[13 19 31]
 [ 0 58  2]
 [ 0 14 18]]

--- Evaluating Multi-label Classification ---

Classes for multi-label: ['character attack' 'deflection' 'dismissal' 'emotional appeal' 'evasion'
 'framing the narrative' 'gaslighting' 'guilt tripping' 'minimization'
 'persuasion' 'playing the victim']

Multi-label Performance Metrics:
- Exact Match Ratio (Accuracy): 0.3871
- Precision (Macro Avg): 0.2709
- R

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:895: UserWarning: unknown class(es) ['none'] will be ignored
  warnings.warn(
